<a href="https://colab.research.google.com/github/Cooljoe67/ML_DSAI/blob/main/11b_kelvin_cv_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Validation Strategies

In this notebook, we'll explore different cross-validation (CV) strategies. We will focus primarily on regression tasks and include one classification example to illustrate stratified splitting.

Scikit-learn provides multiple CV splitters, each suited for different scenarios:

* **[`KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)**
* **[`ShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html)**
* **[`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)**
* **[`GroupShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)**
* **[`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)**
* **[`StratifiedShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html)**


We'll cover each of these, discussing when and how to use them, especially within `GridSearchCV` for hyperparameter tuning, `cross_val_score` for quick evaluation ,and `cross-validate` for deep diagnostics.


### Scikit-Learn CV Tools: What to Use When
| Tool       | What it does |When to Use                                                                 |
|------------------|--------------|------------------------------------------------------------------------------|
| `cross_val_score` |    Evaluates a single model configuration and returns a simple array of test scores for a single metric.| For a quick health check
| `cross_validate`  |     Evaluates a single model but returns a rich dictionary of fit/score times, and training scores.| For deep diagnostics. Use it when you need to check for overfitting     |
| `GridSearchCV`   |       Automates the search for the best hyperparameters by running CV | For model optimisation. Use it when you want find the absolute best settings |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Models
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import Ridge

# CV Tools
from sklearn.model_selection import (
    KFold, ShuffleSplit, StratifiedKFold, StratifiedShuffleSplit,
    GroupKFold, GroupShuffleSplit, GridSearchCV, cross_val_score, cross_validate
)

# Datasets
from sklearn.datasets import load_breast_cancer

# Setup visual style & reproducibility
sns.set_theme(style="whitegrid")
np.random.seed(42)

# supress warnings when fitting
import warnings
warnings.filterwarnings('ignore')

---
## 1.&nbsp; Load and Prepare the Datasets 💾

We will use two real-world datasets:
1.  **Regression:** The Concrete Compressive Strength dataset (features like cement, water; target is strength).
2.  **Classification:** The Breast Cancer dataset (30 features; binary target).

In [ ]:
# --- Regression Data ---
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/refs/heads/master/concrete.csv"
concrete = pd.read_csv(url)

X_reg = concrete.drop('strength', axis=1).values
y_reg = concrete['strength'].values
print("Regression Data")
print(f"Concrete dataset shape: {concrete.shape}")

print("----------------------")

# --- Classification Data ---
cancer = load_breast_cancer()
X_clf = cancer.data
y_clf = cancer.target
print("Classification Data")
print(f"Breast Cancer data shape: {X_clf.shape} | Class distribution: {np.bincount(y_clf)}")

---
---
## 2.&nbsp; Understanding Cross-Validation Strategies 🔬

To make the most of our tools, we need to pick the right strategy for our specific problem. We can generally split these into two categories based on whether we're predicting a continuous number (regression) or a category (classification).



### 2.1. General Strategies (Best for Regression)
These methods are our standard options when we're working on regression tasks or datasets where the target distribution is already well-balanced.

* **`KFold`**: This splits our dataset into k roughly equal parts called folds. Each fold takes a turn as the test set, while the others form the training set. By default, it doesn't shuffle the data.
* **`ShuffleSplit`**: This generates a set number of independent train and test splits by randomly shuffling the data first. Unlike KFold, our folds might overlap, so a single sample could appear in multiple test sets across different runs.
* **`GroupKFold`**: This is helpful when our data is naturally grouped, like having multiple readings for the same patient. It ensures that any given group is either entirely in the training set or entirely in the test set, but never both.
* **`GroupShuffleSplit`**: This offers a random partition that respects our groups. We specify the percentage of groups to include in each test split. While test folds might overlap between iterations, each individual split keeps our group boundaries intact.

### 2.3. Stratified Strategies (Best for Classification)
When we're predicting classes, we need to ensure our training and testing sets look like our original data. These are the go-to choices for classification.

* **`StratifiedKFold`**: This is a version of KFold that preserves the class proportions in every fold. It's essential for classification to make sure each fold is a representative sample of our target classes.
* **`StratifiedShuffleSplit`**: This merges the logic of StratifiedKFold and ShuffleSplit. It gives us the benefit of random, independent splits while still keeping our class proportions consistent across every iteration.

### 2.1&nbsp; Visualising the Split Logic
To build intuition, let's visualise how each CV splitter divides a toy dataset of 100 samples with:
* 3 uneven classes (10%, 30%, 60%)
* 5 randomly assigned groups

In [ ]:
# @title Toy Dataset

# Generate Toy Dataset
n_samples = 100
X_toy = np.random.randn(n_samples, 5)

# Uneven classes (10, 30, 60)
y_toy = np.array([0]*10 + [1]*30 + [2]*60)

# 5 random groups
groups_toy = np.random.randint(0, 5, n_samples)

# Shuffle data
perm = np.random.permutation(n_samples)
y_toy, groups_toy = y_toy[perm], groups_toy[perm]

# --- Improved Visualisation Function ---
def plot_cv_indices(cv, X, y, group, ax, n_splits, lw=10):
    """Create a sample index plot for a specific CV strategy."""
    cmap_data = plt.cm.Paired
    cmap_cv = plt.cm.coolwarm

    for ii, (tr, tt) in enumerate(cv.split(X=X, y=y, groups=group)):
        # Fill in indices with the training/test colours
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1  # Test
        indices[tr] = 0  # Train
        ax.scatter(range(len(indices)), [ii + .5] * len(indices),
                   c=indices, marker='_', lw=lw, cmap=cmap_cv, vmin=-.2, vmax=1.2)

    # Plot the data classes and groups at the end
    ax.scatter(range(len(X)), [ii + 1.5] * len(X), c=y, marker='_', lw=lw, cmap=cmap_data)
    if group is not None:
        ax.scatter(range(len(X)), [ii + 2.5] * len(X), c=group, marker='_', lw=lw, cmap=cmap_data)

    # Formatting
    yticklabels = list(range(n_splits)) + ['class', 'group'] if group is not None else list(range(n_splits)) + ['class']
    ax.set(yticks=np.arange(len(yticklabels)) + .5, yticklabels=yticklabels,
           xlabel='Sample index', ylabel="CV iteration", title=type(cv).__name__)

    return ax

In [ ]:
# @title Visualising the toy dataset

# -- VISUALISE ---

# Define the strategies to compare
cv_strategies = [
    KFold(n_splits=4, shuffle=False),
    ShuffleSplit(n_splits=4, test_size=0.25, random_state=42),
    StratifiedKFold(n_splits=4, shuffle=False),
    StratifiedShuffleSplit(n_splits=4, test_size=0.25, random_state=42),
    GroupKFold(n_splits=4),
    GroupShuffleSplit(n_splits=4, test_size=0.25, random_state=42)
]

# Plotting
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, cv in zip(axes.ravel(), cv_strategies):
    # Only pass groups if the strategy is group-based
    grp = groups_toy if "Group" in type(cv).__name__ else None
    plot_cv_indices(cv, X_toy, y_toy, grp, ax, 4)

# Legend
legend_elements = [Patch(facecolor=plt.cm.coolwarm(.1), label='Training Set'),
                   Patch(facecolor=plt.cm.coolwarm(.9), label='Testing Set')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize='large', bbox_to_anchor=(0.5, 1.05))

plt.tight_layout()
plt.show()

---
## 3.&nbsp; Using Cross-Validation in Model Evaluation 🧮

The `cross_val_score` function automates splitting, training, and scoring.

### 3.1&nbsp; `cross_val_score()`
By default, Scikit-learn uses `KFold` for regression tasks and `StratifiedKFold` for classification tasks. Let's look at both.

In [ ]:
# Regression: Default defaults to KFold
ridge = Ridge(alpha=1.0)
scores_reg_default = cross_val_score(ridge, X_reg, y_reg, scoring='neg_mean_squared_error')
scores_reg = cross_val_score(ridge, X_reg, y_reg, cv=KFold(n_splits=5), scoring='neg_mean_squared_error')
print(f"Ridge default CV MSE: {-scores_reg_default.round(2)}")
print(f"Ridge 5-fold CV MSE:  {-scores_reg.round(2)}")

For classification, `StratifiedKFold` should have been used by default.

To confirm, let's manually compare with an explicit `KFold`:

In [ ]:
# Classification: Default defaults to StratifiedKFold
dt_clf = DecisionTreeClassifier(random_state=42)
scores_clf_default = cross_val_score(dt_clf, X_clf, y_clf, scoring='accuracy')
scores_clf = cross_val_score(dt_clf, X_clf, y_clf, cv=StratifiedKFold(n_splits=5), scoring='accuracy')
print(f"DecisionTree default CV accuracies: {scores_clf_default.round(3)}")
print(f"DecisionTree 5-fold CV accuracies:  {scores_clf.round(3)}")

>By default, `KFold` and `StratifiedKFold` do not shuffle the data before splitting it. This can lead to misleading results if the dataset has any kind of implicit order. For example, if rows are sorted by target value, category, or date.

### 3.2&nbsp; The Danger of Not Shuffling
If your data has an implicit order (e.g., sorted by target value or date), a default `KFold` (which sets `shuffle=False`) will yield disastrous results.

Let's simulate this by sorting the **Concrete** dataset by strength.

In [ ]:
# Sort data by target
sorted_indices = np.argsort(y_reg)
X_sorted, y_sorted = X_reg[sorted_indices], y_reg[sorted_indices]

# Compare Unshuffled vs Shuffled
kf_no_shuffle = KFold(n_splits=5, shuffle=False)
kf_shuffle = KFold(n_splits=5, shuffle=True, random_state=42)

scores_no_shuffle = cross_val_score(ridge, X_sorted, y_sorted, cv=kf_no_shuffle, scoring='neg_mean_squared_error')
scores_shuffle = cross_val_score(ridge, X_sorted, y_sorted, cv=kf_shuffle, scoring='neg_mean_squared_error')

print(f"MSE without shuffling: {-scores_no_shuffle.round(2)}")
print(f"MSE with shuffling:    {-scores_shuffle.round(2)}")

>Without shuffling, KFold might place mostly low values in one fold and high values in another. This gives the model an unrealistic advantage or disadvantage, depending on the fold, which skews performance estimates.

In [ ]:
# @title Visualising the code

# -- VISUALISE ---

# Sort data by target
sorted_indices = np.argsort(y_reg)
X_sorted, y_sorted = X_reg[sorted_indices], y_reg[sorted_indices]

# Define CVs
kf_no_shuffle = KFold(n_splits=5, shuffle=False)
kf_shuffle = KFold(n_splits=5, shuffle=True, random_state=42)

# --- VISUALISE THE FOLDS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, cv, title in zip(axes, [kf_no_shuffle, kf_shuffle], ['Unshuffled KFold (Bad)', 'Shuffled KFold (Good)']):
    for i, (train_idx, test_idx) in enumerate(cv.split(X_sorted)):
        ax.scatter(test_idx, y_sorted[test_idx], label=f'Fold {i+1}', alpha=0.7, s=15)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Sample Index (Sorted by Target)")
    if ax == axes[0]: ax.set_ylabel("Target Value (Concrete Strength)")
    ax.legend(title="Test Set Folds")

plt.tight_layout()
plt.show()

# Print Scores
scores_no_shuffle = cross_val_score(ridge, X_sorted, y_sorted, cv=kf_no_shuffle, scoring='neg_mean_squared_error')
scores_shuffle = cross_val_score(ridge, X_sorted, y_sorted, cv=kf_shuffle, scoring='neg_mean_squared_error')

print(f"Avg MSE (Unshuffled): {-scores_no_shuffle.mean():.2f}")
print(f"Avg MSE (Shuffled):   {-scores_shuffle.mean():.2f}")

### 3.3&nbsp; `cross_validate()` for Deeper Insights
If you need to check for overfitting, use `cross_validate()` to extract both training and testing scores with `return_train_score = True`.

In [ ]:
cv_results = cross_validate(ridge, X_reg, y_reg, cv=5,
                            scoring='neg_mean_squared_error',
                            return_train_score=True
                            )

print(f"Avg Train MSE: {-(cv_results['train_score']).mean():.2f}")
print(f"Avg Test MSE:  {-(cv_results['test_score']).mean():.2f}")
print("-----------")
# Show training and test errors per fold
print("Train scores (MSE):", -cv_results['train_score'])
print("Test scores  (MSE):", -cv_results['test_score'])

> A large gap between training and test scores usually means your model is overfitting; it's learning the training data too well but struggling to generalise.

In [ ]:
# @title Visualising the code
#
# # -- VISUALISE ---

train_mse = -cv_results['train_score']
test_mse = -cv_results['test_score']

# --- VISUALISE TRAIN VS TEST ERROR ---
x = np.arange(5)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
rects1 = ax.bar(x - width/2, train_mse, width, label='Train MSE (How well it memorised)', color='skyblue')
rects2 = ax.bar(x + width/2, test_mse, width, label='Test MSE (How well it generalised)', color='salmon')

ax.set_ylabel('Mean Squared Error')
ax.set_title('Train vs Test Error per Fold (Ridge Regression)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.legend()

plt.show()

print(f"Avg Train MSE: {train_mse.mean():.2f}")
print(f"Avg Test MSE:  {test_mse.mean():.2f}")

> **Observation:** Notice how wildly the Test MSE fluctuates compared to the Train MSE. In Fold 1, the model overfits and struggles massively on the test data (MSE ~209), but in Fold 5, the test data happens to be very "easy" to predict (MSE ~75). This perfectly highlights the core value of cross-validation: a single train/test split relies heavily on luck, but evaluating across 5 folds reveals the model's true average reliability.

---
## 4.&nbsp; Hyperparameter Tuning with GridSearchCV 📊

`GridSearchCV` accepts any of the CV splitters we discussed via its `cv` parameter. Let's look at examples across our datasets.

### 4.1&nbsp; Regression (KFold & ShuffleSplit)

We'll tune a `DecisionTreeRegressor` on the concrete dataset. Hyperparameters to tune: `max_depth` and `min_samples_split`. We'll use 5-fold CV (which will default to regular KFold since it's a regressor).

Perhaps our data has an underlying order. We can use `ShuffleSplit` to randomly sample multiple train/test splits for evaluation. This might give a more robust estimate if data order or particular fold-outs might bias results.

We'll reuse the `params` above but now use `ShuffleSplit` for CV. For fair comparison, we'll do 5 splits each with 20% test.

In [ ]:
# Shared Grid for Regression
reg_params = {
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

dtr = DecisionTreeRegressor(random_state=42)

# --- 1. KFold (Default) ---
grid_kf = GridSearchCV(dtr,
                       param_grid=reg_params,
                       cv=5,
                       scoring='neg_root_mean_squared_error',
                        return_train_score=True)

grid_kf.fit(X_reg, y_reg)
print(f"Best params (KFold):        {grid_kf.best_params_}   | RMSE: {-grid_kf.best_score_:.2f}")


# --- 2. ShuffleSplit ---
ss = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
grid_ss = GridSearchCV(dtr,
                       param_grid=reg_params,
                       cv=ss,
                       scoring='neg_root_mean_squared_error',
                        return_train_score=True)

grid_ss.fit(X_reg, y_reg)
print(f"Best params (ShuffleSplit): {grid_ss.best_params_} | RMSE: {-grid_ss.best_score_:.2f}")

> Notice the dramatic difference in both the chosen hyperparameters and the final RMSE! Because the default `KFold` does not shuffle the data, it falls victim to any hidden sorting in the dataset. `ShuffleSplit` randomises the train/test sets in every iteration, uncovering a completely different optimal tree depth and a much more reliable error estimate.

In [ ]:
#  @title Visualising the code

def get_best_fold_scores(grid):
    best_idx = grid.best_index_ # The row index of the winning hyperparameter combo

    # List comprehensions to grab the score for each fold dynamically
    train_scores = [-grid.cv_results_[f'split{i}_train_score'][best_idx] for i in range(grid.n_splits_)]
    test_scores = [-grid.cv_results_[f'split{i}_test_score'][best_idx] for i in range(grid.n_splits_)]

    return train_scores, test_scores

skf_train, skf_test = get_best_fold_scores(grid_kf)
sss_train, sss_test = get_best_fold_scores(grid_ss)

# --- PLOTTING ---
x = np.arange(5)
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Plot StratifiedKFold
axes[0].bar(x - width/2, skf_train, width, label='Train RMSE', color='skyblue')
axes[0].bar(x + width/2, skf_test, width, label='Validation RMSE', color='salmon')
axes[0].set_title('StratifiedKFold: Best Model Fold Scores', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {i+1}' for i in range(5)])
axes[0].set_ylabel('Error(RMSE)')
axes[0].legend()

# Plot StratifiedShuffleSplit
axes[1].bar(x - width/2, sss_train, width, label='Train RMSE', color='skyblue')
axes[1].bar(x + width/2, sss_test, width, label='Validation RMSE', color='salmon')
axes[1].set_title('ShuffleSplit: Best Model Split Scores', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Split {i+1}' for i in range(5)])
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.2&nbsp; Classification (Stratified Splits)
Classification tasks demand balanced classes. `StratifiedKFold` is the default, but `StratifiedShuffleSplit` is highly effective for large datasets where random sampling is preferred.

In [ ]:
# Shared Grid for Classification
clf_params = {
    'max_depth': [3, 5, None],
    'criterion': ['gini', 'entropy']
}

# --- 1. StratifiedKFold (Default) ---
grid_skf = GridSearchCV(dt_clf,
                        param_grid=clf_params,
                        cv=StratifiedKFold(n_splits=5),
                        scoring='accuracy',
                        return_train_score=True)


grid_skf.fit(X_clf, y_clf)
print(f"Best params (StratifiedKFold):        {grid_skf.best_params_} | Acc: {grid_skf.best_score_:.3f}")

# --- 2. StratifiedShuffleSplit ---
sss = StratifiedShuffleSplit(n_splits=5, test_size=0.20,random_state=42)
grid_sss = GridSearchCV(dt_clf,
                        param_grid=clf_params,
                        cv=sss,
                        scoring='accuracy',
                        return_train_score=True)

grid_sss.fit(X_clf, y_clf)
print(f"Best params (StratifiedShuffleSplit): {grid_sss.best_params_} | Acc: {grid_sss.best_score_:.3f}")

> **Observation:** Notice how the two strategies chose completely different tree depths! `StratifiedKFold` favored an unconstrained tree (`max_depth: None`), which is highly prone to overfitting. However, because `StratifiedShuffleSplit` introduces more randomness across its splits, it penalised that overfitting and revealed that a much simpler, shallower tree (`max_depth: 3`) actually generalises better, yielding a higher overall accuracy.

In [ ]:
#  @title Visualising the code

def get_best_fold_scores(grid):
    best_idx = grid.best_index_ # The row index of the winning hyperparameter combo

    # List comprehensions to grab the score for each fold dynamically
    train_scores = [grid.cv_results_[f'split{i}_train_score'][best_idx] for i in range(grid.n_splits_)]
    test_scores = [grid.cv_results_[f'split{i}_test_score'][best_idx] for i in range(grid.n_splits_)]

    return train_scores, test_scores

skf_train, skf_test = get_best_fold_scores(grid_skf)
sss_train, sss_test = get_best_fold_scores(grid_sss)

# --- PLOTTING ---

from matplotlib.ticker import PercentFormatter # 1. Import the formatter
x = np.arange(5)
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Plot StratifiedKFold
axes[0].bar(x - width/2, skf_train, width, label='Train Accuracy', color='skyblue')
axes[0].bar(x + width/2, skf_test, width, label='Validation Accuracy', color='salmon')
axes[0].set_title('StratifiedKFold: Best Model Fold Scores', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {i+1}' for i in range(5)])
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))

# Plot StratifiedShuffleSplit
axes[1].bar(x - width/2, sss_train, width, label='Train Accuracy', color='skyblue')
axes[1].bar(x + width/2, sss_test, width, label='Validation Accuracy', color='salmon')
axes[1].set_title('StratifiedShuffleSplit: Best Model Split Scores', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Split {i+1}' for i in range(5)])
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.3&nbsp; Handling Grouped Data (GroupKFold)
Imagine our **Concrete** dataset represented tests from different labs, or we wanted to group the data by "Age" (days cured). We must ensure data from the same "Age Group" doesn't bleed across the train/test barrier.

>Note: We use `pd.cut` below to cleanly categorise the continuous age feature into 3 discrete groups.

In [ ]:
# Create 3 groups based on Age: Young (0-7), Mid (8-28), Mature (>28)
concrete['age_group'] = pd.cut(concrete['age'], bins=[0, 7, 28, np.inf], labels=[1, 2, 3]).astype(int)
groups = concrete['age_group'].values

print(f"Group sizes: \n{concrete['age_group'].value_counts().sort_index().to_dict()}")

In [ ]:
concrete

In [ ]:
# --- 1. Data Leakage ---
grid_leakage = GridSearchCV(dtr,
                            param_grid=reg_params,
                            cv=3,
                            scoring='neg_mean_absolute_error',

                            )

# Fit
grid_leakage.fit(X_reg, y_reg)
print(f"Best params (Data Leakage): {grid_leakage.best_params_} | MAE: {-grid_leakage.best_score_:.2f}")


# --- 2. GroupKFold ---
gkf = GroupKFold(n_splits=3)
grid_gkf = GridSearchCV(dtr,
                        param_grid=reg_params,
                        cv=gkf,
                        scoring='neg_mean_absolute_error',

                        )

# Fit (Note the inclusion of the `groups` parameter)
grid_gkf.fit(X_reg, y_reg, groups=groups)
print(f"Best params (GroupKFold):   {grid_gkf.best_params_} | MAE: {-grid_gkf.best_score_:.2f}")

> **Observation:** Notice how the MAE is artificially lower in the "Data Leakage" example; this happens because the standard K-Fold allows data from the same age group to bleed across the train and test sets, whereas `GroupKFold` gives a more realistic (higher) error by forcing the model to predict on completely unseen groups.

> **Note:** In this simulation, we binned the continuous "Age" variable to create groups. Because `GroupKFold` holds out an entire group, our model was forced to predict the strength of an **entirely unseen age bracket** (extrapolation), which is why the error is higher. In the real world, groups are typically categorical identifiers like `Patient_ID`, `Lab_Session`, or `Website_User`, ensuring that no individual subject leaks across the train/test barrier!


In [ ]:
#  @title Visualising the code

# Helper function to extract the fold scores for the winning hyperparameter combo
def get_best_fold_maes(grid):
    best_idx = grid.best_index_
    # We negate the scores because scikit-learn returns negative MAE
    fold_scores = [-grid.cv_results_[f'split{i}_test_score'][best_idx] for i in range(grid.n_splits_)]
    return fold_scores

# Extract the fold scores for both models
mae_leakage_folds = get_best_fold_maes(grid_leakage)
mae_gkf_folds = get_best_fold_maes(grid_gkf)

# --- VISUALISING THE PER-FOLD SCORES ---
x = np.arange(3)  # We used 3 folds
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

# Plot the grouped bars
ax.bar(x - width/2, mae_leakage_folds, width, label='Standard CV (Data Leakage)', color='salmon')
ax.bar(x + width/2, mae_gkf_folds, width, label='GroupKFold (No Leakage)', color='skyblue')

# Add labels and formatting
ax.set_ylabel('Mean Absolute Error (MAE)')
ax.set_title('Per-Fold MAE: Standard CV vs GroupKFold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(3)])
ax.legend()

plt.tight_layout()
plt.show()

### 4.4&nbsp; GroupKFold vs. GroupShuffleSplit
`GroupKFold` has a randomised cousin, `GroupShuffleSplit`.

* `GroupKFold` guarantees that every group will be in the test set exactly once. If you have 3 groups, you can only do 3 splits.
* `GroupShuffleSplit` allows you to decouple the number of splits from the number of groups. You specify the `test_size` (the percentage of **groups** to hold out) and can run as many randomized splits as you want.

In [ ]:
# --- 3. GroupShuffleSplit ---
# Let's run 5 randomized iterations, holding out 20% of the groups for testing each time.
gss = GroupShuffleSplit(n_splits=5,  test_size=0.2, random_state=42)

grid_gss = GridSearchCV(dtr,
                        param_grid=reg_params,
                        cv=gss,
                        scoring='neg_mean_absolute_error')

# Fit (Again, we must pass the `groups` parameter)
grid_gss.fit(X_reg, y_reg, groups=groups)
print(f"Best params (GroupShuffleSplit): {grid_gss.best_params_} | MAE: {-grid_gss.best_score_:.2f}")

> **Observation:** Notice how we were able to run 5 independent cross-validation splits even though our dataset only contains 3 age groups; `GroupShuffleSplit` gives you the flexibility to randomly hold out a percentage of *groups* (rather than individual samples), completely decoupling your iteration count from your group count.

### 4.5&nbsp; The "Why": StratifiedKFold vs. StratifiedShuffleSplit

Why use `StratifiedShuffleSplit` if `StratifiedKFold` already maintains class balance? **Total control over data proportions.**

* **StratifiedKFold:** Test size is strictly tied to the number of splits (e.g., 5 splits strictly mandates a 20% test size). Every sample appears in the test set exactly once.
* **StratifiedShuffleSplit:** Decouples `n_splits` from `test_size`. You can run 50 iterations with a 20% test size, or 5 iterations with a 50% test size.

**Best Use Cases:**
1. **Massive Datasets:** Run just 2 or 3 splits with a tiny fraction of the data (e.g., `train_size=0.01`, `test_size=0.005`) to save compute time.
2. **Robustness Testing:** Run 100+ randomized iterations to aggressively test model stability.

Let's look at an extreme example: running 20 iterations, but only using 15% of the data for training and 10% for testing (ignoring 75% of the dataset to save compute time).

In [ ]:
# This datset isn't huge, so we will slightly increase the test/train sizes
# 20 independent splits, but only using 15% of data for training and 10% for testing per split
extreme_sss = StratifiedShuffleSplit(n_splits=20, train_size=0.15, test_size=0.10, random_state=42)

grid_extreme = GridSearchCV(dt_clf,
                            param_grid=clf_params,
                            cv=extreme_sss,
                            scoring='accuracy')

# Fit on the classification data
grid_extreme.fit(X_clf, y_clf)

print(f"Best params (Extreme SSS): {grid_extreme.best_params_}")
print(f"Best CV accuracy (20 splits): {grid_extreme.best_score_:.3f}")



> **Observation:** Notice how we successfully completed a robust 20-iteration search while completely ignoring 75% of the data in each split; this highlights the unique power of `StratifiedShuffleSplit` to decouple your cross-validation loops from your train/test sizes, saving massive computation time on large datasets.

---
## 5.&nbsp; Choosing the Right Cross-Validation Strategy 🧭

| Cross-Validation Strategy       | When to Use                                                                 |
|--------------------------------|------------------------------------------------------------------------------|
| `KFold`                        | General-purpose, especially for regression tasks or perfectly balanced classification |
| `StratifiedKFold`              | **Standard for Classification.** Maintains class proportions in folds.       |
| `ShuffleSplit`                 | When you want random train/test splits, or want to control train/test sizes independent of iteration count. |
| `StratifiedShuffleSplit`       | Like `ShuffleSplit`, but maintains class proportions                        |
| `GroupKFold`                   | **Crucial for nested/grouped data** (e.g., multiple scans from the same patient). Prevents data leakage. |
| `GroupShuffleSplit`            | Randomized version of group-aware splitting.                                |



---
## 6.&nbsp; Challenges 🙂

To solidify your understanding, try the following challenges:

1. **KFold vs. ShuffleSplit Experiment**: On a regression dataset of your choice, compare model performance using KFold and ShuffleSplit (with a similar number of iterations). Do they yield different mean scores? Discuss why or why not.
2. **Stratification Importance**: [Create an imbalanced binary classification dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html) (e.g., 95% of class 0, 5% of class 1). Evaluate a classifier with 5-fold KFold vs 5-fold StratifiedKFold. What do you observe in the per-fold scores?
3. **The Grouping Edge-Case**: For GroupKFold, simulate a case where each group has just 1 or 2 samples. What issues arise?
4. **Data Leakage**: Create a sorted dataset (e.g., sorted by target or time), and test KFold with and without shuffling. What differences do you see?

Happy learning and experimenting! Remember, the right CV strategy is just as important as the model you choose to train.